# 07 · Внешняя оценка CNN на собственных аудиофайлах

CPU или GPU. Используется только `best_model.pth`; обучение и подбор порога здесь не выполняются. Основной результат — на уровне **исходного файла**. Положительный класс: **FAKE = 1**.

In [ ]:
# 1. Подключение проекта и зависимостей
import subprocess
import sys
from pathlib import Path

repo = Path('/content/audio-deepfake-detector')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/Unvand/audio-deepfake-detector.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-colab-train.txt')], check=True)
sys.path.insert(0, str(repo))
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Git commit:', commit)

In [ ]:
# 2. Подключение Google Drive и создание пустых папок
from google.colab import drive, output
from IPython.display import Markdown, display

from audio_deepfake_detector.src.external_evaluation import ensure_external_folders

drive.mount('/content/drive')
output.enable_custom_widget_manager()
external_root = Path('/content/drive/MyDrive/VKR_RuASD/external_test_v1')
folders = ensure_external_folders(external_root)
for label, path in folders.items():
    print(f'{label}: {path}')
print('Откройте Google Drive → Мой диск → VKR_RuASD → external_test_v1 → input.')
print('Загрузите настоящую речь в REAL, синтезированную или преобразованную — в FAKE.')

In [ ]:
# 3. Проверка входных файлов и checkpoint; выполните снова после загрузки аудио
import pandas as pd

from audio_deepfake_detector.src.cnn_training import resolve_run_directory
from audio_deepfake_detector.src.external_evaluation import scan_external_files

files, scan_errors = scan_external_files(external_root)
if not scan_errors.empty:
    display(Markdown('### Файлы с ошибкой декодирования'))
    display(scan_errors)
if files.empty:
    print('Аудиофайлов пока нет. Загрузите файлы в REAL и FAKE, затем повторите эту ячейку.')
else:
    summary = files.groupby('true_class').agg(Количество_файлов=('source_file_id', 'size'),
                                              Общая_длительность_сек=('duration_seconds', 'sum'))
    summary = summary.reindex(['REAL', 'FAKE'], fill_value=0)
    display(summary)
    if summary['Количество_файлов'].max() > 2 * max(1, summary['Количество_файлов'].min()):
        print('Внимание: число REAL и FAKE существенно различается.')
    duplicates = files.loc[files.duplicated('sha256', keep=False)].sort_values('sha256')
    if not duplicates.empty:
        print('Повторяющееся содержимое внутри одного класса:')
        display(duplicates[['relative_path', 'sha256']])
default_checkpoint = resolve_run_directory(external_root.parents[0] / 'training_v1', 'baseline_cached_v1') / 'best_model.pth'
print('Checkpoint по умолчанию:', default_checkpoint)
print('Доступен' if default_checkpoint.is_file() else 'Пока не найден')

In [ ]:
# 4. Настройки оценки
import ipywidgets as widgets
import torch

from audio_deepfake_detector.src.ruasd_evaluation import load_best_model

run_id = widgets.Text(value='baseline_cached_v1', description='run_id')
batch_size = widgets.BoundedIntText(value=32, min=1, max=256, description='Batch size')
save_results = widgets.Checkbox(value=True, description='Сохранять результаты')
local_copy = widgets.Checkbox(value=False, description='Копировать оригиналы во временный кэш /content')
display(widgets.VBox([run_id, batch_size, save_results, local_copy]))
print('Устройство:', 'GPU' if torch.cuda.is_available() else 'CPU')
print('Временные FLAC-сегменты всегда создаются в /content и удаляются после оценки.')

In [ ]:
# 5. Синхронная внешняя оценка. Переключите на True и запустите ▶
import matplotlib.pyplot as plt

from audio_deepfake_detector.src.external_evaluation import (
    evaluate_external,
    external_figures,
    external_summary,
    save_external,
    scan_external_files,
)

RUN_EXTERNAL_EVALUATION = False

if RUN_EXTERNAL_EVALUATION:
    run = run_id.value.strip()
    checkpoint = resolve_run_directory(external_root.parents[0] / 'training_v1', run) / 'best_model.pth'
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Не найден checkpoint: {checkpoint}')
    files, scan_errors = scan_external_files(external_root)
    if files.empty:
        raise ValueError('Нет доступных аудиофайлов в REAL и FAKE.')
    print(f'Проверено {len(files)} файлов. Ошибок чтения: {len(scan_errors)}.')
    model_check, mel_check, threshold_check, saved_check = load_best_model(
        checkpoint, expected_epoch=8 if run == 'baseline_cached_v1' else None)
    del model_check
    print('Mel:', mel_check, '· порог из checkpoint:', threshold_check)
    preprocessing_path = external_root.parents[0] / 'prepared_v1' / 'preprocessing_config.json'
    if not preprocessing_path.is_file():
        raise FileNotFoundError(f'Не найдена конфигурация подготовки: {preprocessing_path}')
    result = evaluate_external(files, checkpoint, preprocessing_path, batch_size=batch_size.value,
                               local_copy=local_copy.value, git_commit=commit)
    result['errors'] = pd.concat([scan_errors, result['errors']], ignore_index=True)
    if not result['errors'].empty:
        print(f'Внимание: {len(result["errors"])} файлов не вошли в оценку. Список будет сохранён в decode_errors.csv.')
        display(result['errors'])
    figures = external_figures(result)
    saved_to = None
    if save_results.value:
        saved_to = save_external(result, figures, folders['results'])
        print('Результаты:', saved_to)
    print('Оценка завершена. Теперь выполните ячейки 6–8.')

In [ ]:
# 6. Результаты по исходным файлам и графики
if 'result' not in globals():
    print('Сначала выполните оценку в ячейке 5.')
else:
    display(result['files'][['relative_path', 'true_class', 'fake_score',
                             'predicted_class', 'correct', 'n_segments']]
            .rename(columns={'relative_path': 'Файл', 'true_class': 'Истинный класс',
                             'fake_score': 'FAKE-score', 'predicted_class': 'Прогноз',
                             'correct': 'Верно', 'n_segments': 'Сегментов'}))
    display(pd.DataFrame(result['metrics']).T.fillna('N/A'))
    display(Markdown('### Результаты по подпапкам (если они созданы)'))
    display(result['group_metrics'])
    display(Markdown(external_summary(result)))
    for figure in figures.values():
        display(figure)
        plt.close(figure)

In [ ]:
# 7. Ошибки, прослушивание и log-Mel выбранных сегментов
import tempfile

from IPython.display import Audio

from audio_deepfake_detector.src.cnn_training import MelConfig, RuASDSegmentDataset
from audio_deepfake_detector.src.external_evaluation import (
    decode_audio,
    prepare_segments,
)

SHOW_ERROR_EXAMPLES = False

if SHOW_ERROR_EXAMPLES:
    if 'result' not in globals():
        raise RuntimeError('Сначала выполните оценку в ячейке 5.')
    errors = result['files'].loc[~result['files']['correct']]
    for title, actual in (('REAL → FAKE', 0), ('FAKE → REAL', 1)):
        selected = errors.loc[errors['true_label'].eq(actual)].head(3)
        display(Markdown(f'### {title} · {len(errors.loc[errors["true_label"].eq(actual)])} ошибок'))
        display(selected)
        for item in selected.itertuples(index=False):
            waveform, sample_rate = decode_audio(Path(item.source_path))
            display(Audio(waveform[:30 * sample_rate], rate=sample_rate, normalize=False))
            with tempfile.TemporaryDirectory() as temporary:
                source_row = files.loc[files['source_file_id'].eq(item.source_file_id)]
                segment_rows, failures = prepare_segments(
                    source_row, Path(temporary), sample_rate=result['config']['mel_config']['sample_rate'])
                if not failures.empty:
                    display(failures)
                    continue
                dataset = RuASDSegmentDataset(segment_rows.head(1), temporary, 'test_unseen',
                                               MelConfig(**result['config']['mel_config']))
                mel_db, _ = dataset.visualization_item(0)
                fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
                duration = mel_db.shape[-1] * dataset.mel_config.hop_length / dataset.mel_config.sample_rate
                image = ax.imshow(mel_db.numpy(), origin='lower', aspect='auto',
                                  extent=(0, duration, 0, dataset.mel_config.n_mels), cmap='magma')
                ax.set(title=f'Log-Mel первого сегмента: {item.relative_path}',
                       xlabel='Время, с', ylabel='Mel-полоса')
                fig.colorbar(image, ax=ax, label='дБ')
                display(fig)
                plt.close(fig)
    print('Одна спектрограмма не доказывает причину ошибки модели.')

## 8. Сохранение и журнал ВКР

При включённом «Сохранять результаты» каждый запуск создаёт новый каталог в `MyDrive/VKR_RuASD/external_test_v1/results/`. В нём находятся CSV, JSON, PNG и `evaluation_summary.md`. Оригиналы REAL/FAKE остаются в `input/`.

Скачайте `evaluation_summary.md`, проверьте отчёт и из корня репозитория выполните:

```bash
python scripts/build_vkr_report.py --append-evaluation /путь/к/evaluation_summary.md
```

Затем сохраните обновлённые `docs/VKR_EXPERIMENT_LOG.md` и `docs/VKR_EXPERIMENT_REPORT.docx` в Git. RuASD validation, RuASD test_unseen и внешний пользовательский тест — разные этапы исследования.